In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge 

In [2]:
df_train = pd.read_csv('../datasets/benchmark/data_train_benchmark_lag.csv')
df_test = pd.read_csv('../datasets/benchmark/data_test_benchmark_lag.csv')

In [3]:
df_train.info()
df_train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7368 entries, 0 to 7367
Columns: 514 entries, timestamp to output_hybrid_lag_72
dtypes: float64(511), int64(2), object(1)
memory usage: 28.9+ MB


,timestamp,temperature,humidity,surface_radiation,upper_atmospheric_radiation,hour,month,air_density,wind_velocity,output_hybrid,...,output_hybrid_lag_63,output_hybrid_lag_64,output_hybrid_lag_65,output_hybrid_lag_66,output_hybrid_lag_67,output_hybrid_lag_68,output_hybrid_lag_69,output_hybrid_lag_70,output_hybrid_lag_71,output_hybrid_lag_72
0,2019-01-04 08:00:00,-14.599,0.001,72.765,214.723,8,1,1.322,5.683,1042.513,...,691.624,596.188,585.535,1183.530,1861.908,2466.902,2643.376,2418.649,1940.806,1461.490
1,2019-01-04 09:00:00,-13.791,0.001,145.273,351.385,9,1,1.325,5.768,1276.272,...,719.946,691.624,596.188,585.535,1183.530,1861.908,2466.902,2643.376,2418.649,1940.806
2,2019-01-04 10:00:00,-12.917,0.001,198.601,436.617,10,1,1.328,6.510,1959.656,...,773.037,719.946,691.624,596.188,585.535,1183.530,1861.908,2466.902,2643.376,2418.649
3,2019-01-04 11:00:00,-12.343,0.001,217.455,464.588,11,1,1.330,7.488,2685.257,...,751.143,773.037,719.946,691.624,596.188,585.535,1183.530,1861.908,2466.902,2643.376
4,2019-01-04 12:00:00,-12.295,0.001,200.135,433.386,12,1,1.333,8.004,2829.141,...,521.661,751.143,773.037,719.946,691.624,596.188,585.535,1183.530,1861.908,2466.902


In [4]:
feature_base = [col for col in df_train.columns if 'lag' not in col]
print(f"number of feature: {len(feature_base)}")
print(feature_base)

feature_lag = [col for col in df_train.columns if 'lag' in col]
print(f"number of feature lag: {len(feature_lag)}")

number of feature: 10
['timestamp', 'temperature', 'humidity', 'surface_radiation', 'upper_atmospheric_radiation', 'hour', 'month', 'air_density', 'wind_velocity', 'output_hybrid']
number of feature lag: 504


In [5]:
target = 'output_hybrid'
kolom_yang_dibuang = [target, 'timestamp'] 
fitur_aktif_total = [col for col in df_train.columns if col not in kolom_yang_dibuang]

num_lag_only = len([c for c in fitur_aktif_total if 'lag' in c])

y_train = df_train[target]
y_test = df_test[target]

def calculate_maape(y_true, y_pred):
    epsilon = 1e-10
    return np.mean(np.arctan(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon)))) * 100

list_model = {
    'RidgeRegression': Ridge(alpha=0.1, solver='lsqr'), 
    'RandomForest': RandomForestRegressor(
        n_estimators=100, 
        max_depth=None, 
        random_state=42, 
        n_jobs=-1
    ),
    
    'XGBoost': XGBRegressor(
        n_estimators=150, 
        learning_rate=0.05, 
        max_depth=6, 
        random_state=42, 
        n_jobs=-1
    ),
    
    'LightGBM': LGBMRegressor(
        n_estimators=150, 
        learning_rate=0.1, 
        max_depth=6, 
        random_state=42, 
        n_jobs=-1, 
        verbose=-1
    ), 
    
    'SVR': SVR(
        kernel='rbf', 
        C=10.0, 
        epsilon=0.01, 
        gamma=0.01, 
        
    )
}

results_records = []

print(f"Memulai training multi-model dengan TOTAL {len(fitur_aktif_total)} FITUR...")

for nama_model, regressor in list_model.items():
    
    X_train = df_train[fitur_aktif_total]
    X_test = df_test[fitur_aktif_total]
    
    scaler_X = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    
    print(f"-> Melatih {nama_model} (Total Lag Fitur: {num_lag_only})...")
    regressor.fit(X_train_scaled, y_train_scaled)

    preds_scaled = regressor.predict(X_test_scaled)
    preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    maape = calculate_maape(y_test, preds)
    r2 = r2_score(y_test, preds)

    results_records.append({
        'Model': nama_model,
        'Num_Lag_Features': num_lag_only, 
        'RMSE': rmse,
        'MAE': mae, 
        'MAAPE (%)': maape, 
        'R2': r2,
    })

df_result_summary = pd.DataFrame(results_records)

print("\n========================================================================================")
print("                                RINGKASAN EVALUASI PERFORMA MODEL                       ")
print("========================================================================================")
print(df_result_summary.to_string(index=False, formatters={
    'RMSE': '{:,.4f}'.format, 
    'MAE': '{:,.4f}'.format,
    'MAAPE (%)': '{:,.2f}%'.format,
    'R2': '{:,.4f}'.format
}))
print("========================================================================================")

Memulai training multi-model dengan TOTAL 512 FITUR...
-> Melatih RidgeRegression (Total Lag Fitur: 504)...
-> Melatih RandomForest (Total Lag Fitur: 504)...
-> Melatih XGBoost (Total Lag Fitur: 504)...
-> Melatih LightGBM (Total Lag Fitur: 504)...


C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


-> Melatih SVR (Total Lag Fitur: 504)...

                                RINGKASAN EVALUASI PERFORMA MODEL                       
          Model  Num_Lag_Features     RMSE      MAE MAAPE (%)     R2
RidgeRegression               504 221.2103 156.0273    21.18% 0.9775
   RandomForest               504 275.3524 131.7361     9.62% 0.9652
        XGBoost               504 250.1357 129.4541    12.54% 0.9713
       LightGBM               504 245.4748 132.6983    13.25% 0.9723
            SVR               504 276.4170 206.0908    26.14% 0.9649
